# LPM_transplant — 01_dapi_whole_organoid_mask_review

**Feeds:** Fig 5n

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


            # 01 | DAPI Whole-Organoid Mask Review

            ## Notebook Scope

            This notebook runs the current DAPI-only whole-organoid masking scaffold and reviews representative mask overlays before interpretation of donor/host identities.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from skimage import measure

from scripts import run_pixel_level_quantification as rpq
from scripts import transplant_quantification_helpers as tqh

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

            ## Paths And Parameters

            This stage writes the current working tables for masks, thresholds, class summaries, sampled pixels, and density maps.
            It is intentionally a full stage run so notebook `02` can focus on interpretation rather than recomputation.
            

In [ ]:
DATA_DIR = ROOT / "data"
MANIFEST_OUTPUT = ROOT / "results" / "manifests" / "raw_input_manifest.tsv"
MASK_DIR = ROOT / "results" / "masks" / "dapi_whole_organoid_masks"
PLANE_METRICS_OUTPUT = ROOT / "results" / "tables" / "01_mask_and_plane_metrics.tsv"
NORMALIZATION_OUTPUT = ROOT / "results" / "tables" / "01b_condition_channel_normalization.tsv"
NORMALIZATION_HIST_OUTPUT = ROOT / "results" / "tables" / "01b_condition_channel_normalization_histograms.tsv"
THRESHOLD_OUTPUT = ROOT / "results" / "tables" / "02_ratio_thresholds.tsv"
CLASS_SUMMARY_OUTPUT = ROOT / "results" / "tables" / "02_pixel_class_summary.tsv"
FOXF1_IDENTITY_OUTPUT = ROOT / "results" / "tables" / "02_foxf1_positive_identity_summary.tsv"
PIXEL_SAMPLE_OUTPUT = ROOT / "results" / "tables" / "02_sampled_pixel_profiles.tsv"
DENSITY_OUTPUT = ROOT / "results" / "tables" / "02_density_maps.tsv"

THRESHOLD_METHOD = "log1p_otsu"
THRESHOLD_Z = 4.0
DOMINANCE_MARGIN_LOG2 = 0.75
THRESHOLD_REFERENCE_CONDITION = "ctrl"
HOST_THRESHOLD_SCOPE = "condition"
DONOR_THRESHOLD_SCOPE = "condition"
FOXF1_THRESHOLD_SCOPE = "condition"
MESP2_THRESHOLD_SCOPE = "global"
HOST_THRESHOLD_SCALE = 0.85
DONOR_THRESHOLD_SCALE = 0.85
FOXF1_THRESHOLD_SCALE = 0.85
MESP2_THRESHOLD_SCALE = 1.0
FOXF1_THRESHOLD_CONDITION_SCALES = {}
FOXF1_THRESHOLD_CONDITION_VALUES = {}
FOXF1_THRESHOLD_FILE_SCALES = {
    # Example: 4: 1.15 would raise the FOXF1 threshold for file_id 4 by 15%.
}
FOXF1_COMPONENT_RADIUS_PX = 80.0
FOXF1_HOST_SEED_THRESHOLD_SCALE = 1.15
NORMALIZATION_SAMPLE_PER_PLANE = 50000

WRITE_OUTPUTS = True
RUN_PIPELINE = True

print("DATA_DIR:", DATA_DIR)
print("MASK_DIR:", MASK_DIR)
print("NORMALIZATION_OUTPUT:", NORMALIZATION_OUTPUT)
print("NORMALIZATION_HIST_OUTPUT:", NORMALIZATION_HIST_OUTPUT)
print("THRESHOLD_METHOD:", THRESHOLD_METHOD)
print("THRESHOLD_REFERENCE_CONDITION:", THRESHOLD_REFERENCE_CONDITION)
print("HOST_THRESHOLD_SCOPE:", HOST_THRESHOLD_SCOPE)
print("DONOR_THRESHOLD_SCOPE:", DONOR_THRESHOLD_SCOPE)
print("FOXF1_THRESHOLD_SCOPE:", FOXF1_THRESHOLD_SCOPE)
print("HOST_THRESHOLD_SCALE:", HOST_THRESHOLD_SCALE)
print("DONOR_THRESHOLD_SCALE:", DONOR_THRESHOLD_SCALE)
print("FOXF1_THRESHOLD_SCALE:", FOXF1_THRESHOLD_SCALE)
print("FOXF1_THRESHOLD_CONDITION_SCALES:", FOXF1_THRESHOLD_CONDITION_SCALES)
print("FOXF1_THRESHOLD_CONDITION_VALUES:", FOXF1_THRESHOLD_CONDITION_VALUES)
print("FOXF1_THRESHOLD_FILE_SCALES:", FOXF1_THRESHOLD_FILE_SCALES)
print("FOXF1_COMPONENT_RADIUS_PX:", FOXF1_COMPONENT_RADIUS_PX)
print("FOXF1_HOST_SEED_THRESHOLD_SCALE:", FOXF1_HOST_SEED_THRESHOLD_SCALE)
print("NORMALIZATION_SAMPLE_PER_PLANE:", NORMALIZATION_SAMPLE_PER_PLANE)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)

In [ ]:
if RUN_PIPELINE:
    quant_res = rpq.run_quantification_pipeline(
        root=ROOT,
        data_dir=DATA_DIR,
        manifest_output=MANIFEST_OUTPUT,
        mask_dir=MASK_DIR,
        plane_metrics_output=PLANE_METRICS_OUTPUT,
        normalization_output=NORMALIZATION_OUTPUT,
        normalization_hist_output=NORMALIZATION_HIST_OUTPUT,
        threshold_output=THRESHOLD_OUTPUT,
        class_summary_output=CLASS_SUMMARY_OUTPUT,
        foxf1_identity_output=FOXF1_IDENTITY_OUTPUT,
        pixel_sample_output=PIXEL_SAMPLE_OUTPUT,
        density_output=DENSITY_OUTPUT,
        threshold_method=THRESHOLD_METHOD,
        threshold_z=THRESHOLD_Z,
        dominance_margin_log2=DOMINANCE_MARGIN_LOG2,
        normalization_sample_per_plane=NORMALIZATION_SAMPLE_PER_PLANE,
        threshold_reference_condition=THRESHOLD_REFERENCE_CONDITION,
        host_threshold_scope=HOST_THRESHOLD_SCOPE,
        donor_threshold_scope=DONOR_THRESHOLD_SCOPE,
        foxf1_threshold_scope=FOXF1_THRESHOLD_SCOPE,
        mesp2_threshold_scope=MESP2_THRESHOLD_SCOPE,
        host_threshold_scale=HOST_THRESHOLD_SCALE,
        donor_threshold_scale=DONOR_THRESHOLD_SCALE,
        foxf1_threshold_scale=FOXF1_THRESHOLD_SCALE,
        mesp2_threshold_scale=MESP2_THRESHOLD_SCALE,
        foxf1_threshold_condition_scales=FOXF1_THRESHOLD_CONDITION_SCALES,
        foxf1_threshold_condition_values=FOXF1_THRESHOLD_CONDITION_VALUES,
        foxf1_threshold_file_scales=FOXF1_THRESHOLD_FILE_SCALES,
        foxf1_component_radius_px=FOXF1_COMPONENT_RADIUS_PX,
        foxf1_host_seed_threshold_scale=FOXF1_HOST_SEED_THRESHOLD_SCALE,
        write_outputs=WRITE_OUTPUTS,
    )
    manifest_df = quant_res["manifest_df"].copy()
    plane_metrics_df = quant_res["plane_metrics_df"].copy()
    normalization_df = quant_res["normalization_df"].copy()
    normalization_hist_df = quant_res["normalization_hist_df"].copy()
else:
    manifest_df = pd.read_csv(MANIFEST_OUTPUT, sep="\t")
    plane_metrics_df = pd.read_csv(PLANE_METRICS_OUTPUT, sep="\t")
    normalization_df = pd.read_csv(NORMALIZATION_OUTPUT, sep="\t")
    normalization_hist_df = pd.read_csv(NORMALIZATION_HIST_OUTPUT, sep="\t")

print("Manifest rows:", len(manifest_df))
print("Plane rows:", len(plane_metrics_df))
display(plane_metrics_df.head(20))
display(normalization_df)

In [ ]:
mask_summary = (
    plane_metrics_df.groupby(["condition", "file_id"], as_index=False)
    .agg(
        file_name=("file_name", "first"),
        z_planes=("z_index", "nunique"),
        mean_mask_fraction=("mask_fraction", "mean"),
        min_mask_fraction=("mask_fraction", "min"),
        max_mask_fraction=("mask_fraction", "max"),
        mean_dapi_floor=("dapi_floor", "mean"),
    )
    .sort_values(["condition", "file_id"])
)
display(mask_summary)

In [ ]:
def _robust_rescale(image: np.ndarray, q_low: float = 0.01, q_high: float = 0.99) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo = float(np.quantile(finite, q_low))
    hi = float(np.quantile(finite, q_high))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def plot_mask_review_panels(example_df: pd.DataFrame) -> None:
    if example_df.empty:
        print("No example rows selected.")
        return
    fig, axes = plt.subplots(len(example_df), 3, figsize=(12, 4 * len(example_df)))
    if len(example_df) == 1:
        axes = np.asarray([axes])

    for ax_row, row in zip(axes, example_df.itertuples(index=False)):
        stack = tqh.load_transplant_stack(ROOT / str(row.file_path))
        bf_idx = tqh.channel_index(stack.canonical_channel_names, "brightfield")
        dapi_idx = tqh.channel_index(stack.canonical_channel_names, "dapi")
        bf = np.asarray(stack.data_czyx[bf_idx, int(row.z_index)], dtype=np.float32)
        dapi = np.asarray(stack.data_czyx[dapi_idx, int(row.z_index)], dtype=np.float32)
        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)

        bf_disp = _robust_rescale(bf)
        dapi_disp = _robust_rescale(dapi)
        sample_label = f"{row.condition} | {row.position_label} | z{row.z_index}"

        ax_row[0].imshow(bf_disp, cmap="gray")
        ax_row[0].set_title(f"{sample_label}\nBF")
        ax_row[0].axis("off")

        ax_row[1].imshow(dapi_disp, cmap="gray")
        ax_row[1].set_title(f"{sample_label}\nDAPI | thr={row.threshold_value:.1f}")
        ax_row[1].axis("off")

        ax_row[2].imshow(dapi_disp, cmap="gray")
        for contour in measure.find_contours(mask.astype(np.uint8), level=0.5):
            ax_row[2].plot(contour[:, 1], contour[:, 0], color="#00e676", linewidth=1.4)
        ax_row[2].set_title(f"{sample_label}\nMask boundary | frac={row.mask_fraction:.3f}")
        ax_row[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
representative_per_file = (
    plane_metrics_df.assign(
        z_center_distance=lambda df: (
            df["z_index"] - df.groupby("file_id")["z_index"].transform("median")
        ).abs()
    )
    .sort_values(["condition", "file_id", "z_center_distance", "z_index"])
    .groupby("file_id", as_index=False)
    .head(1)
)

example_rows = representative_per_file.sort_values(["condition", "file_id", "z_index"]).reset_index(drop=True)

display(
    example_rows[
        ["condition", "position_label", "file_name", "z_index", "mask_fraction", "dapi_floor", "threshold_value", "mask_path"]
    ]
)
plot_mask_review_panels(example_rows)

            ## Top 3 Most Deviant Z Planes

            These are the 3 planes with the largest within-stack deviation in `mask_fraction`, so the worst apparent DAPI-mask outliers can be inspected directly.
            

In [ ]:
mask_deviants = (
    plane_metrics_df.assign(
        mask_fraction_stack_mean=lambda df: df.groupby("file_id")["mask_fraction"].transform("mean"),
        mask_fraction_stack_std=lambda df: df.groupby("file_id")["mask_fraction"].transform("std"),
    )
    .assign(
        mask_fraction_deviation_z=lambda df: (
            (df["mask_fraction"] - df["mask_fraction_stack_mean"])
            / df["mask_fraction_stack_std"].replace(0, np.nan)
        ),
    )
    .assign(
        abs_mask_fraction_deviation_z=lambda df: df["mask_fraction_deviation_z"].abs(),
    )
    .sort_values(
        ["abs_mask_fraction_deviation_z", "condition", "file_id", "z_index"],
        ascending=[False, True, True, True],
    )
    .head(3)
    .reset_index(drop=True)
)

display(
    mask_deviants[
        [
            "condition",
            "position_label",
            "file_name",
            "z_index",
            "mask_fraction",
            "mask_fraction_deviation_z",
            "dapi_floor",
            "threshold_value",
            "mask_path",
        ]
    ]
)
plot_mask_review_panels(mask_deviants)

            ## All Z Planes For `1(z4)(trunk with CFP)(from 260227).czi`

            This section shows every z plane for that control stack so the full DAPI-mask behavior can be checked, not just the representative plane.
            

In [ ]:
target_position_label = "1(z4)(trunk with CFP)(from 260227)"
target_rows = (
    plane_metrics_df[plane_metrics_df["position_label"].astype(str) == target_position_label]
    .sort_values("z_index")
    .reset_index(drop=True)
)

display(
    target_rows[
        ["condition", "position_label", "file_name", "z_index", "mask_fraction", "dapi_floor", "threshold_value", "mask_path"]
    ]
)
plot_mask_review_panels(target_rows)